# Week 3 Practical: Model Evaluation & Interpretability
**AI for Drug Discovery**

In this practical, you will:
1. Implement scaffold splitting for molecular data
2. Compare random vs. scaffold split performance
3. Compute classification and regression metrics
4. Use SHAP to interpret your QSAR model
5. Define an applicability domain

In [ ]:
# Install dependencies
!pip install rdkit-pypi scikit-learn xgboost shap pandas matplotlib seaborn -q

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Scaffolds
from rdkit.Chem.Scaffolds import MurckoScaffold
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_squared_error, r2_score, roc_auc_score,
                             precision_recall_curve, roc_curve, average_precision_score,
                             classification_report)
from sklearn.model_selection import train_test_split
import shap

sns.set_style('whitegrid')
print('All imports successful!')

## 1. Load Dataset and Generate Features
We'll reuse the Delaney solubility dataset from Week 2.

In [ ]:
# Load Delaney dataset
try:
    url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'
    df = pd.read_csv(url)
except:
    url = 'https://raw.githubusercontent.com/PatWalters/datafiles/main/delaney.csv'
    df = pd.read_csv(url)

smiles_col = [c for c in df.columns if 'smiles' in c.lower()][0]
target_col = [c for c in df.columns if 'solubility' in c.lower() or 'log' in c.lower()][0]

df['mol'] = df[smiles_col].apply(Chem.MolFromSmiles)
df = df[df['mol'].notna()].reset_index(drop=True)
print(f'Loaded {len(df)} valid molecules')

In [ ]:
# Generate features (fingerprints + descriptors)
def mol_to_fp(mol, radius=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros(n_bits, dtype=np.int8)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def calc_descriptors(mol):
    return [Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
            Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
            Descriptors.NumAromaticRings(mol), Descriptors.HeavyAtomCount(mol),
            Descriptors.RingCount(mol), Descriptors.FractionCSP3(mol)]

desc_names = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'AromaticRings',
              'HeavyAtoms', 'RingCount', 'FractionCSP3']

fp_array = np.array([mol_to_fp(m) for m in df['mol']])
desc_array = np.array([calc_descriptors(m) for m in df['mol']])
X = np.hstack([fp_array, desc_array])
y = df[target_col].values

feature_names = [f'FP_{i}' for i in range(2048)] + desc_names
print(f'Feature matrix: {X.shape}, Target: {y.shape}')

## 2. Scaffold Splitting
Implement Bemis-Murcko scaffold decomposition and split by scaffold.

**Reference:** Bemis, G.W. & Murcko, M.A. (1996). The Properties of Known Drugs. J. Med. Chem. 39:2887-2893

In [ ]:
# Get Bemis-Murcko scaffolds
def get_scaffold(mol):
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except:
        return ''

df['scaffold'] = df['mol'].apply(get_scaffold)
print(f'Unique scaffolds: {df["scaffold"].nunique()} / {len(df)} molecules')
print(f'Top 5 scaffolds by frequency:')
print(df['scaffold'].value_counts().head())

In [ ]:
# Scaffold-based train/test split
def scaffold_split(df, test_frac=0.2, random_state=42):
    scaffolds = df['scaffold'].values
    unique_scaffolds = list(set(scaffolds))
    np.random.seed(random_state)
    np.random.shuffle(unique_scaffolds)

    test_size = int(len(df) * test_frac)
    test_indices = []
    for scaffold in unique_scaffolds:
        indices = np.where(scaffolds == scaffold)[0].tolist()
        test_indices.extend(indices)
        if len(test_indices) >= test_size:
            break

    test_idx = set(test_indices)
    train_idx = [i for i in range(len(df)) if i not in test_idx]
    return train_idx, list(test_idx)

train_idx_scaffold, test_idx_scaffold = scaffold_split(df)
print(f'Scaffold split - Train: {len(train_idx_scaffold)}, Test: {len(test_idx_scaffold)}')

## 3. Compare Random vs. Scaffold Split

In [ ]:
# Random split
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scaffold split
X_train_scaf = X[train_idx_scaffold]
X_test_scaf = X[test_idx_scaffold]
y_train_scaf = y[train_idx_scaffold]
y_test_scaf = y[test_idx_scaffold]

# Train RF on both splits
rf_rand = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rf_rand.fit(X_train_rand, y_train_rand)
y_pred_rand = rf_rand.predict(X_test_rand)

rf_scaf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rf_scaf.fit(X_train_scaf, y_train_scaf)
y_pred_scaf = rf_scaf.predict(X_test_scaf)

print('Random Split:')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test_rand, y_pred_rand)):.3f}')
print(f'  R2:   {r2_score(y_test_rand, y_pred_rand):.3f}')
print()
print('Scaffold Split:')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test_scaf, y_pred_scaf)):.3f}')
print(f'  R2:   {r2_score(y_test_scaf, y_pred_scaf):.3f}')
print()
print('Notice how scaffold split gives lower (more realistic) performance!')

In [ ]:
# Visualize the comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, yt, yp, title in zip(axes,
    [y_test_rand, y_test_scaf], [y_pred_rand, y_pred_scaf],
    ['Random Split', 'Scaffold Split']):
    ax.scatter(yt, yp, alpha=0.5, s=30)
    ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2 = r2_score(yt, yp)
    ax.set_title(f'{title}\nRMSE={rmse:.3f}, R2={r2:.3f}', fontsize=13)
    ax.set_xlabel('Actual logS')
    ax.set_ylabel('Predicted logS')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 4. SHAP Analysis
Use SHAP (SHapley Additive exPlanations) to understand what features drive predictions.

**Reference:** Lundberg, S.M. & Lee, S. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS.

In [ ]:
# SHAP analysis on the scaffold-split model
# Use a subset for speed
X_explain = X_test_scaf[:100]

explainer = shap.TreeExplainer(rf_scaf)
shap_values = explainer.shap_values(X_explain)

print(f'SHAP values shape: {shap_values.shape}')
print('SHAP analysis complete!')

In [ ]:
# SHAP summary plot (beeswarm) - focusing on descriptor features
# For readability, show only the descriptor features (last 10)
desc_shap = shap_values[:, -len(desc_names):]
desc_X = X_explain[:, -len(desc_names):]

plt.figure(figsize=(10, 6))
shap.summary_plot(desc_shap, desc_X, feature_names=desc_names, show=False)
plt.title('SHAP Feature Importance (Descriptors)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP waterfall plot for a single prediction
idx = 0  # First test molecule
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap.Explanation(
    values=desc_shap[idx],
    base_values=explainer.expected_value,
    data=desc_X[idx],
    feature_names=desc_names
), show=False)
plt.title(f'SHAP Explanation for Molecule {idx}', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Applicability Domain
Define the chemical space where our model is reliable.

In [ ]:
from rdkit import DataStructs

# Compute Tanimoto distance from each test molecule to nearest training molecule
train_fps = [AllChem.GetMorganFingerprintAsBitVect(df['mol'].iloc[i], 2, nBits=2048)
             for i in train_idx_scaffold]

def max_tanimoto_to_train(mol, train_fps):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    sims = [DataStructs.TanimotoSimilarity(fp, tfp) for tfp in train_fps]
    return max(sims)

test_mols = [df['mol'].iloc[i] for i in test_idx_scaffold]
max_sims = [max_tanimoto_to_train(m, train_fps) for m in test_mols[:100]]

# Plot: similarity vs prediction error
errors = np.abs(y_test_scaf[:100] - y_pred_scaf[:100])

plt.figure(figsize=(10, 6))
plt.scatter(max_sims, errors, alpha=0.5, s=40)
plt.xlabel('Max Tanimoto Similarity to Training Set', fontsize=12)
plt.ylabel('Absolute Prediction Error (logS)', fontsize=12)
plt.title('Applicability Domain: Similarity vs. Error', fontsize=14)
plt.axvline(0.4, color='red', linestyle='--', label='AD threshold = 0.4')
plt.legend()
plt.tight_layout()
plt.show()

# Statistics
in_ad = np.array(max_sims) >= 0.4
print(f'Molecules in AD (similarity >= 0.4): {in_ad.sum()} / {len(max_sims)}')
print(f'RMSE in AD: {np.sqrt(np.mean(errors[in_ad]**2)):.3f}')
if (~in_ad).sum() > 0:
    print(f'RMSE outside AD: {np.sqrt(np.mean(errors[~in_ad]**2)):.3f}')

## 6. Exercises

1. **Temporal split**: If the dataset had dates, how would you implement a temporal split?
2. **Different SHAP**: Use KernelSHAP instead of TreeSHAP. How do results compare?
3. **Classification**: Convert solubility to binary (soluble/insoluble at logS > -2). Compute ROC-AUC and PR-AUC.
4. **Challenge**: Implement conformal prediction intervals for your solubility model.

## References
- Bemis, G.W. & Murcko, M.A. (1996). The Properties of Known Drugs. J. Med. Chem. 39:2887-2893
- Lundberg, S.M. & Lee, S. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS
- Wallach, I. & Heifets, A. (2018). Most Ligand-Based Virtual Screening Benchmarks Reward Memorization. JCIM
- Sahigara, F. et al. (2012). Comparison of Different Approaches to Define the Applicability Domain. Molecules 17:4791-4810